## לינאריזציה: דעיכה מעריכית דרך ln

`linear_fit` יודעת להתאים רק **קווים ישרים**. אבל הרבה תופעות פיזיקליות אינן ליניאריות -- למשל **בליעה** של קרינה דרך חומר, לפי חוק בير-למברט: $I(x) = I_0 e^{-\mu x}$, כאשר $I$ עוצמת הקרינה, $x$ עובי הבולם, $\mu$ מקדם הבליעה.

הטריק: מפעילים `ln` על שני האגפים -- $\ln I = \ln I_0 - \mu x$ -- וזה **קו ישר** ב-$x$ מול $\ln I$! שיפוע הקו הוא $-\mu$, החיתוך הוא $\ln I_0$. זו "לינאריזציה" -- הופכים בעיה לא-ליניארית לליניארית על ידי שינוי משתנה.

In [1]:
import numpy as np

rng = np.random.default_rng(3)
I0_true = 120.0
mu_true = 0.35

x_abs = np.arange(9, dtype=float)                       # עובי הבולם, יחידות שרירותיות
I_true = I0_true * np.exp(-mu_true * x_abs)
I_meas = I_true + rng.normal(0, 3.0, size=len(x_abs))    # מדידה עם רעש

print(np.round(I_meas, 2))

[126.12  76.9   60.84  40.29  28.23  20.21   8.63   9.66   4.7 ]


### לינאריזציה והתאמה

In [2]:
def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    m = np.sum((x - x_bar) * (y - y_bar)) / np.sum((x - x_bar)**2)
    b = y_bar - m * x_bar
    return m, b

y_lin = np.log(I_meas)          # ln(I)
m, b = linear_fit(x_abs, y_lin)

mu_fit = -m
I0_fit = np.exp(b)
print(f"mu משוחזר = {mu_fit:.3f}  (אמיתי: {mu_true})")
print(f"I0 משוחזר = {I0_fit:.1f}  (אמיתי: {I0_true})")

mu משוחזר = 0.400  (אמיתי: 0.35)
I0 משוחזר = 128.3  (אמיתי: 120.0)


### באג נפוץ: ln על ערך שלילי או אפס

אם יש בנתונים ערך שלילי (למשל אחרי חיסור רקע אגרסיבי מדי) או אפס, `np.log` **לא זורק שגיאה** -- הוא מחזיר `nan` (עם `RuntimeWarning` שקל לפספס), וממשיך הלאה בשקט. אם לא בודקים במפורש, ה-`nan` הזה מזהם את `linear_fit` כולה -- וכל התוצאה (`m`, `b`) הופכת ל-`nan`, בלי שום שגיאה שמסבירה למה.

In [3]:
I_meas_bad = I_meas.copy()
I_meas_bad[-1] = I_meas_bad[-1] - 12.0   # לדוגמה: חיסור רקע שגוי הוציא ערך שלילי

y_lin_bad = np.log(I_meas_bad)   # RuntimeWarning, לא Exception
print(y_lin_bad)
print("יש nan בנתונים:", np.isnan(y_lin_bad).any())

m_bad, b_bad = linear_fit(x_abs, y_lin_bad)
print(f"m עם ה-nan בפנים: {m_bad}")   # nan מזהם הכל

[4.8372557  4.34244834 4.10832197 3.69608397 3.34051587 3.00598363
 2.15580206 2.26793498        nan]
יש nan בנתונים: True
m עם ה-nan בפנים: nan


/tmp/ipykernel_4180748/85346611.py:4: RuntimeWarning: invalid value encountered in log
  y_lin_bad = np.log(I_meas_bad)   # RuntimeWarning, לא Exception


### נסו בעצמכם

כתבו קוד שמסנן (`~np.isnan(...)`, או `I_meas_bad > 0`) את הנקודות הבעייתיות **לפני** ה-`log`, ומריץ את ההתאמה מחדש על הנתונים הנקיים בלבד.

In [4]:
# valid = I_meas_bad > 0
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
valid = I_meas_bad > 0
x_valid = x_abs[valid]
y_valid = np.log(I_meas_bad[valid])

m_valid, b_valid = linear_fit(x_valid, y_valid)
print(f"mu משוחזר (אחרי סינון) = {-m_valid:.3f}")
```
`````

### בדקו את עצמכם

In [5]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "למה `np.log` על ערך שלילי מסוכן יותר מ-Exception רגיל?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הוא מחזיר nan בשקט (רק אזהרה) והריצה ממשיכה, כך שהבעיה מתגלה רק בתוצאה הסופית - אם בכלל", "correct": True, "feedback": "נכון - זו בדיוק ההגדרה של 'באג סמנטי'."},
            {"answer": "כי הוא תמיד קורס את כל הפייתון", "correct": False, "feedback": "לא - הפעולה עצמה לא קורסת."},
            {"answer": "np.log על ערך שלילי לא באמת בעייתי, זו רק אזהרת סגנון", "correct": False, "feedback": "לא נכון - התוצאה (nan) לא שמישה, זו בעיה אמיתית."},
            {"answer": "כי הוא מוחק את המערך המקורי מהזיכרון", "correct": False, "feedback": "לא — np.log לא נוגע במערך המקורי בכלל; הוא רק מחזיר nan עבור הערכים השליליים בפלט החדש."}
        ]
    }
]
display_quiz(questions)

<IPython.core.display.Javascript object>

### תרגול עצמי

חשבו $R^2$ עבור ההתאמה הליניארית (בעולם ה-`ln`, כלומר על `x_abs` מול `y_lin` המקוריים, בלי הנקודה הבעייתית) וגם, בנפרד, "תרגמו" את התחזית חזרה לעולם המקורי -- חשבו את `I_pred = I0_fit * np.exp(-mu_fit * x_abs)` והשוו גרפית (או במספרים) ל-`I_meas`.

In [6]:
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
y_pred_lin = m*x_abs + b
resid_lin = y_lin - y_pred_lin
ss_res = np.sum(resid_lin**2)
ss_tot = np.sum((y_lin - y_lin.mean())**2)
r2_lin = 1 - ss_res/ss_tot
print(f"R^2 (במרחב ln): {r2_lin:.4f}")

I_pred = I0_fit * np.exp(-mu_fit * x_abs)
print("מדוד: ", np.round(I_meas, 1))
print("מודל: ", np.round(I_pred, 1))
```
`````